# Step 1b: Query Engine 심화 — 비용 추적, 스트리밍, 메시지 관리

## 학습 목표

- API 호출 비용을 실시간으로 추적하는 **CostTracker**를 이해하고 구현합니다
- **스트리밍 API**의 내부 동작과 이벤트 처리 방식을 분석합니다
- 대화 히스토리가 길어질 때 **메시지 truncation** 전략을 배웁니다

> Step 1에서 에이전트 루프의 핵심 패턴을 배웠습니다. 이 Step에서는 루프를 실전에서 운용하기 위한 세 가지 핵심 인프라를 추가합니다.

---

## Section 1: 소스 분석 — 비용 추적 (CostTracker)

### 1-1. 모델별 가격 정의 (src/utils/modelCost.ts)

Claude Code는 모델별로 입력/출력/캐시 토큰의 가격을 정의합니다. 가격은 **1M 토큰당 USD** 단위입니다:

```typescript
// src/utils/modelCost.ts:27-88 — 모델별 가격 정의

export type ModelCosts = {
  inputTokens: number           // 입력 토큰 가격 ($ per 1M tokens)
  outputTokens: number          // 출력 토큰 가격 ($ per 1M tokens)
  promptCacheWriteTokens: number // 캐시 쓰기 가격
  promptCacheReadTokens: number  // 캐시 읽기 가격 (입력 대비 90% 할인)
  webSearchRequests: number      // 웹 검색 요청 가격
}

// Sonnet 계열: $3 input / $15 output per Mtok
export const COST_TIER_3_15 = {
  inputTokens: 3,
  outputTokens: 15,
  promptCacheWriteTokens: 3.75,
  promptCacheReadTokens: 0.3,     // ★ 입력($3) 대비 90% 할인
  webSearchRequests: 0.01,
}

// Opus 4/4.1: $15 input / $75 output per Mtok
export const COST_TIER_15_75 = {
  inputTokens: 15,
  outputTokens: 75,
  promptCacheWriteTokens: 18.75,
  promptCacheReadTokens: 1.5,     // ★ 입력($15) 대비 90% 할인
  webSearchRequests: 0.01,
}

// Haiku 3.5: $0.80 input / $4 output per Mtok
export const COST_HAIKU_35 = {
  inputTokens: 0.8,
  outputTokens: 4,
  promptCacheWriteTokens: 1,
  promptCacheReadTokens: 0.08,
  webSearchRequests: 0.01,
}
```

### 1-2. 비용 계산 공식 (src/utils/modelCost.ts:131-141)

```typescript
function tokensToUSDCost(modelCosts: ModelCosts, usage: Usage): number {
  return (
    (usage.input_tokens / 1_000_000) * modelCosts.inputTokens +
    (usage.output_tokens / 1_000_000) * modelCosts.outputTokens +
    ((usage.cache_read_input_tokens ?? 0) / 1_000_000) *
      modelCosts.promptCacheReadTokens +          // ★ 캐시 읽기 할인
    ((usage.cache_creation_input_tokens ?? 0) / 1_000_000) *
      modelCosts.promptCacheWriteTokens
  )
}
```

### 1-3. /cost 커맨드 — 세션 비용 리포트 (src/cost-tracker.ts)

Claude Code의 `/cost` 커맨드는 `formatTotalCost()`를 호출하여 세션 전체 비용을 보여줍니다:

```typescript
// src/cost-tracker.ts:228-256 — formatTotalCost()
export function formatTotalCost(): string {
  return chalk.dim(
    `Total cost:            ${formatCost(getTotalCostUSD())}\n` +
    `Total duration (API):  ${formatDuration(getTotalAPIDuration())}\n` +
    `Total duration (wall): ${formatDuration(getTotalDuration())}\n` +
    `Total code changes:    ${lines_added} lines added, ${lines_removed} lines removed\n` +
    modelUsageDisplay   // ★ 모델별 토큰 사용량과 비용 내역
  )
}
```

모델별 사용량은 `addToTotalSessionCost()`가 매 API 응답마다 누적합니다:
- 입력/출력 토큰 수
- 캐시 읽기/쓰기 토큰 수
- 누적 비용 (USD)

**핵심 설계**: 캐시 읽기 토큰은 입력 대비 90% 할인됩니다. 시스템 프롬프트가 동일하면 캐시에서 읽히므로, 대화가 길어질수록 캐시 절감 효과가 커집니다.

---

## Section 2: Python 구현 — CostTracker

In [ ]:
import sys, os
sys.path.insert(0, ".")

from dataclasses import dataclass, field
from typing import Optional


# ─── 모델 가격 정의 ───────────────────────────────────────────────
# src/utils/modelCost.ts의 MODEL_COSTS에 대응

MODEL_PRICING = {
    'claude-opus-4-20250514':       {'input': 15.0,  'output': 75.0,
                                     'cache_read': 1.5, 'cache_write': 18.75},
    'claude-sonnet-4-20250514':     {'input': 3.0,   'output': 15.0,
                                     'cache_read': 0.3, 'cache_write': 3.75},
    'claude-haiku-3-5-20241022':    {'input': 0.80,  'output': 4.0,
                                     'cache_read': 0.08, 'cache_write': 1.0},
    'gpt-4o':                       {'input': 2.50,  'output': 10.0,
                                     'cache_read': 0.0, 'cache_write': 0.0},
    'gpt-4o-mini':                  {'input': 0.15,  'output': 0.60,
                                     'cache_read': 0.0, 'cache_write': 0.0},
}


# ─── 사용량 레코드 ─────────────────────────────────────────────────

@dataclass
class UsageRecord:
    """한 번의 API 호출에서 발생한 토큰 사용량"""
    input_tokens: int
    output_tokens: int
    cache_read_tokens: int = 0
    cache_creation_tokens: int = 0


# ─── CostTracker ───────────────────────────────────────────────────

class CostTracker:
    """
    세션 전체의 API 호출 비용을 추적합니다.
    
    Claude Code의 cost-tracker.ts:addToTotalSessionCost()에 대응합니다.
    매 API 호출 후 add_usage()를 호출하면 누적 비용을 계산합니다.
    """

    def __init__(self, model: str):
        self.model = model
        self.pricing = MODEL_PRICING.get(model, {'input': 3.0, 'output': 15.0,
                                                  'cache_read': 0.3, 'cache_write': 3.75})
        self.records: list[UsageRecord] = []
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_cache_read_tokens = 0
        self.total_cache_creation_tokens = 0

    def add_usage(self, record: UsageRecord) -> dict:
        """사용량을 추가하고 이번 턴의 비용을 반환합니다."""
        self.records.append(record)
        self.total_input_tokens += record.input_tokens
        self.total_output_tokens += record.output_tokens
        self.total_cache_read_tokens += record.cache_read_tokens
        self.total_cache_creation_tokens += record.cache_creation_tokens

        turn_cost = self._calculate_cost(record)
        return {
            'turn': len(self.records),
            'input_tokens': record.input_tokens,
            'output_tokens': record.output_tokens,
            'cache_read_tokens': record.cache_read_tokens,
            'turn_cost_usd': turn_cost,
            'total_cost_usd': self.get_total_cost(),
        }

    def _calculate_cost(self, record: UsageRecord) -> float:
        """
        비용 계산 — src/utils/modelCost.ts:tokensToUSDCost()에 대응
        
        공식: (tokens / 1,000,000) * price_per_mtok
        """
        return (
            (record.input_tokens / 1_000_000) * self.pricing['input'] +
            (record.output_tokens / 1_000_000) * self.pricing['output'] +
            (record.cache_read_tokens / 1_000_000) * self.pricing['cache_read'] +
            (record.cache_creation_tokens / 1_000_000) * self.pricing['cache_write']
        )

    def get_total_cost(self) -> float:
        """누적 총 비용 (USD)"""
        return sum(self._calculate_cost(r) for r in self.records)

    def get_summary(self) -> dict:
        """전체 사용량 요약"""
        return {
            'model': self.model,
            'turns': len(self.records),
            'total_input_tokens': self.total_input_tokens,
            'total_output_tokens': self.total_output_tokens,
            'total_cache_read_tokens': self.total_cache_read_tokens,
            'total_cache_creation_tokens': self.total_cache_creation_tokens,
            'total_cost_usd': self.get_total_cost(),
            'cache_savings_usd': self._calculate_cache_savings(),
        }

    def _calculate_cache_savings(self) -> float:
        """
        캐시로 절약한 비용 계산.
        캐시 읽기가 없었다면 입력 토큰 가격이 적용되었을 것이므로,
        (입력 가격 - 캐시 읽기 가격) * 캐시 읽기 토큰이 절약액입니다.
        """
        price_diff = self.pricing['input'] - self.pricing['cache_read']
        return (self.total_cache_read_tokens / 1_000_000) * price_diff

    def format_report(self) -> str:
        """
        사람이 읽기 좋은 리포트 — src/cost-tracker.ts:formatTotalCost()에 대응
        """
        summary = self.get_summary()
        cost = summary['total_cost_usd']
        cost_str = f"${cost:.2f}" if cost > 0.5 else f"${cost:.4f}"

        lines = [
            f"{'=' * 50}",
            f"  Model:              {summary['model']}",
            f"  Turns:              {summary['turns']}",
            f"  Input tokens:       {summary['total_input_tokens']:,}",
            f"  Output tokens:      {summary['total_output_tokens']:,}",
            f"  Cache read tokens:  {summary['total_cache_read_tokens']:,}",
            f"  Cache write tokens: {summary['total_cache_creation_tokens']:,}",
            f"  ──────────────────────────────────────────────",
            f"  Total cost:         {cost_str}",
            f"  Cache savings:      ${summary['cache_savings_usd']:.4f}",
            f"{'=' * 50}",
        ]
        return '\n'.join(lines)


print("CostTracker 클래스 정의 완료")

In [ ]:
# ─── 데모: 5턴 대화 비용 추적 시뮬레이션 ─────────────────────────

tracker = CostTracker(model='claude-sonnet-4-20250514')

# 5턴의 시뮬레이션 사용량 — 대화가 진행될수록 입력 토큰이 증가 (히스토리 누적)
simulated_turns = [
    UsageRecord(input_tokens=1_200,  output_tokens=350),
    UsageRecord(input_tokens=2_800,  output_tokens=420,  cache_read_tokens=1_000),
    UsageRecord(input_tokens=5_500,  output_tokens=800,  cache_read_tokens=2_500),
    UsageRecord(input_tokens=9_200,  output_tokens=600,  cache_read_tokens=4_800,
                cache_creation_tokens=200),
    UsageRecord(input_tokens=12_000, output_tokens=1_500, cache_read_tokens=7_000),
]

print("=== 턴별 비용 추적 ===")
print(f"{'턴':>3}  {'입력':>8}  {'출력':>7}  {'캐시읽기':>8}  {'턴 비용':>10}  {'누적 비용':>10}")
print("-" * 65)

for record in simulated_turns:
    result = tracker.add_usage(record)
    print(
        f"{result['turn']:>3}  "
        f"{result['input_tokens']:>8,}  "
        f"{result['output_tokens']:>7,}  "
        f"{result['cache_read_tokens']:>8,}  "
        f"${result['turn_cost_usd']:>8.4f}  "
        f"${result['total_cost_usd']:>8.4f}"
    )

print()
print(tracker.format_report())

**관찰 포인트:**
- 대화가 길어질수록 **입력 토큰이 급증**합니다 (히스토리 전체를 매번 전송)
- **캐시 읽기**가 증가하면서 비용 증가를 일부 상쇄합니다
- 출력 토큰 가격이 입력보다 5배 비싸므로, 짧은 응답이 비용 효율적입니다
- 이것이 Step 8(컨텍스트 압축)이 필요한 이유입니다 — 히스토리를 줄이면 비용도 줄어듭니다

---

## Section 3: 소스 분석 — 스트리밍 내부 동작

### 3-1. 스트리밍 이벤트 흐름

Claude Code는 모든 API 호출을 **스트리밍 모드**로 수행합니다 (`stream=True`). 응답이 완성될 때까지 기다리지 않고, 토큰이 생성될 때마다 실시간으로 처리합니다.

스트리밍 이벤트의 전체 흐름:

```
API call (stream=True)
    |
    v
message_start                          -- 메시지 시작
    |
    v
content_block_start (type=text)        -- 텍스트 블록 시작
    |
content_block_delta (text="파일을")     -- 텍스트 조각 (실시간 UI 표시)
content_block_delta (text=" 읽겠")     -- ...
content_block_delta (text="습니다")    -- ...
    |
content_block_stop                     -- 텍스트 블록 종료
    |
    v
content_block_start (type=tool_use)    -- 도구 호출 블록 시작
    |
content_block_delta (input JSON 조각)  -- 도구 입력을 점진적으로 수신
content_block_delta (input JSON 조각)  -- ...
    |
content_block_stop                     -- 도구 호출 블록 종료
    |
    v
message_delta (usage={...})            -- ★ 사용량 정보 (토큰 수)
    |
message_stop                           -- 메시지 완료
```

### 3-2. Claude Code의 스트리밍 처리 (src/query.ts:659)

```typescript
// src/query.ts:659 — 스트리밍 응답 소비
for await (const message of deps.callModel({
  messages,
  system,
  tools,
  stream: true,   // ★ 항상 스트리밍
})) {
  // 텍스트 델타 → 즉시 yield (UI에 실시간 표시)
  // tool_use 블록 → 완전한 JSON이 될 때까지 축적
  // message_delta → usage 통계 추출 → CostTracker에 전달
}
```

**핵심 설계 결정:**
- **텍스트 델타**는 도착 즉시 UI에 전달합니다 (사용자가 실시간으로 응답을 봄)
- **도구 입력 JSON**은 조각이 모두 모여야 유효한 JSON이므로, 블록 완료 후 처리합니다
- **사용량 통계**는 `message_delta` 이벤트에서 도착하며, 이때 CostTracker에 기록합니다

---

## Section 4: Python 구현 — 스트리밍 이벤트 분석기 (StreamAnalyzer)

In [ ]:
import time
import asyncio
from dataclasses import dataclass, field
from typing import AsyncIterator, Any


@dataclass
class StreamEventLog:
    """하나의 스트리밍 이벤트 로그"""
    timestamp: float         # 이벤트 도착 시간 (epoch)
    elapsed_ms: float        # 스트림 시작 이후 경과 시간 (ms)
    event_type: str          # 이벤트 타입
    size: int = 0            # 텍스트 길이 또는 데이터 크기
    detail: str = ""         # 추가 정보


class StreamAnalyzer:
    """
    스트리밍 이벤트를 분석하고 통계를 수집합니다.
    
    Claude Code의 스트리밍 처리 로직(query.ts:659)을 관찰 가능하게 만든 래퍼입니다.
    원본 스트림을 감싸서 각 이벤트의 타이밍과 크기를 기록합니다.
    """

    def __init__(self):
        self.logs: list[StreamEventLog] = []
        self.start_time: float = 0
        self.text_chunks: list[str] = []
        self.tool_events: list[dict] = []

    async def wrap_stream(self, stream: AsyncIterator) -> AsyncIterator:
        """
        스트림을 감싸서 이벤트를 로깅하면서 그대로 전달합니다.
        
        사용법:
            analyzer = StreamAnalyzer()
            async for event in analyzer.wrap_stream(client.stream(...)):
                # 원래대로 이벤트 처리
        """
        self.start_time = time.time()
        self.logs = []
        self.text_chunks = []
        self.tool_events = []

        async for event in stream:
            now = time.time()
            elapsed = (now - self.start_time) * 1000  # ms

            log = StreamEventLog(
                timestamp=now,
                elapsed_ms=elapsed,
                event_type=event.type,
            )

            if event.type == "text_delta":
                log.size = len(event.text)
                log.detail = repr(event.text[:40])
                self.text_chunks.append(event.text)
            elif event.type == "tool_use":
                if event.tool_call:
                    log.detail = f"{event.tool_call.name}({event.tool_call.input})"
                    self.tool_events.append({
                        'name': event.tool_call.name,
                        'input': event.tool_call.input,
                    })
            elif event.type in ("message_start", "message_end"):
                log.detail = event.type

            self.logs.append(log)
            yield event  # 원본 이벤트를 그대로 전달

    def get_stats(self) -> dict:
        """수집된 이벤트 통계를 반환합니다."""
        if not self.logs:
            return {'total_events': 0}

        total_duration = self.logs[-1].elapsed_ms
        event_counts = {}
        for log in self.logs:
            event_counts[log.event_type] = event_counts.get(log.event_type, 0) + 1

        # 첫 텍스트까지의 시간 (Time to First Token)
        ttft = None
        for log in self.logs:
            if log.event_type == "text_delta":
                ttft = log.elapsed_ms
                break

        total_text = "".join(self.text_chunks)

        return {
            'total_events': len(self.logs),
            'event_counts': event_counts,
            'total_duration_ms': round(total_duration, 1),
            'ttft_ms': round(ttft, 1) if ttft else None,
            'text_chunks': len(self.text_chunks),
            'total_text_length': len(total_text),
            'tool_calls': len(self.tool_events),
            'tool_names': [t['name'] for t in self.tool_events],
        }

    def format_timeline(self, max_lines: int = 30) -> str:
        """이벤트 타임라인을 사람이 읽기 좋게 포맷합니다."""
        lines = [f"{'시간(ms)':>10}  {'타입':<20}  {'크기':>5}  {'상세'}"]
        lines.append("-" * 70)

        for i, log in enumerate(self.logs[:max_lines]):
            lines.append(
                f"{log.elapsed_ms:>10.1f}  "
                f"{log.event_type:<20}  "
                f"{log.size:>5}  "
                f"{log.detail[:40]}"
            )

        if len(self.logs) > max_lines:
            lines.append(f"  ... ({len(self.logs) - max_lines}개 이벤트 생략)")

        return "\n".join(lines)


print("StreamAnalyzer 클래스 정의 완료")

In [ ]:
# ─── 데모: 시뮬레이션 스트림으로 StreamAnalyzer 테스트 ──────────────
#
# 실제 API 키가 없어도 동작을 확인할 수 있도록,
# 가짜 스트림을 만들어 StreamAnalyzer를 테스트합니다.

from mini_claude.llm_client import StreamEvent, ToolCall


async def fake_stream():
    """도구 호출이 포함된 가짜 스트리밍 응답을 생성합니다."""
    yield StreamEvent(type="message_start")
    await asyncio.sleep(0.05)  # TTFT 시뮬레이션

    # 텍스트 응답
    for chunk in ["파일을 ", "읽어서 ", "분석하겠", "습니다."]:
        yield StreamEvent(type="text_delta", text=chunk)
        await asyncio.sleep(0.02)  # 토큰 생성 간격

    # 도구 호출
    yield StreamEvent(
        type="tool_use",
        tool_call=ToolCall(id="call_1", name="Read", input={"file_path": "README.md"})
    )
    await asyncio.sleep(0.01)

    yield StreamEvent(type="message_end")


# StreamAnalyzer로 감싸서 분석
analyzer = StreamAnalyzer()

print("=== 스트리밍 이벤트 타임라인 ===")
print()

collected_text = []
async for event in analyzer.wrap_stream(fake_stream()):
    if event.type == "text_delta":
        collected_text.append(event.text)

# 타임라인 출력
print(analyzer.format_timeline())
print()

# 통계 출력
stats = analyzer.get_stats()
print("=== 스트리밍 통계 ===")
print(f"  총 이벤트 수: {stats['total_events']}")
print(f"  이벤트별 카운트: {stats['event_counts']}")
print(f"  전체 소요 시간: {stats['total_duration_ms']}ms")
print(f"  첫 토큰까지 시간 (TTFT): {stats['ttft_ms']}ms")
print(f"  텍스트 청크 수: {stats['text_chunks']}")
print(f"  전체 텍스트: {''.join(collected_text)}")
print(f"  도구 호출 수: {stats['tool_calls']}")
print(f"  호출된 도구: {stats['tool_names']}")

**관찰 포인트:**
- `message_start`부터 첫 `text_delta`까지의 시간이 **TTFT (Time to First Token)**입니다
- 텍스트 청크는 보통 몇 개의 토큰 단위로 도착합니다
- 도구 호출은 하나의 `tool_use` 이벤트로 완성된 상태로 도착합니다 (SDK가 조립)
- `message_end` 이후에 전체 usage 정보를 얻을 수 있습니다

---

## Section 5: 소스 분석 — 메시지 Truncation

### 5-1. 왜 Truncation이 필요한가?

에이전트가 도구를 반복 호출하면 대화 히스토리가 급격히 커집니다:

| 턴 수 | 대략적 토큰 수 | 비용 (Sonnet) |
|-------|---------------|---------------|
| 5     | ~10,000       | ~$0.03        |
| 15    | ~50,000       | ~$0.15        |
| 30    | ~150,000      | ~$0.45        |
| 50+   | 200,000+ (한도 초과!) | ❌ 에러   |

Claude의 컨텍스트 윈도우는 200K 토큰입니다. 히스토리가 이를 초과하면 `prompt_too_long` 에러가 발생합니다.

### 5-2. Claude Code의 계층적 메시지 관리 전략

Claude Code는 세 가지 수준으로 메시지를 관리합니다:

1. **단순 Truncation** (이 Step): 오래된 메시지를 user-assistant 쌍 단위로 제거
2. **Micro Compact** (Step 8): 오래된 도구 결과를 `[cleared]`로 교체
3. **Auto Compact** (Step 8): LLM에게 이전 대화를 요약하도록 요청

### 5-3. 쌍 단위 제거가 중요한 이유

Anthropic API는 메시지 순서에 엄격한 규칙이 있습니다:
- `user` → `assistant` → `user` → `assistant` ... (교대 규칙)
- `tool_result`는 직전의 `tool_use`와 반드시 쌍을 이루어야 함

따라서 메시지를 제거할 때 개별 메시지가 아닌 **user-assistant 쌍 단위**로 제거해야 합니다.
`tool_result` 메시지도 해당 `assistant` (tool_use) 메시지와 함께 제거합니다.

```
제거 가능한 단위:
  [user] + [assistant]                        -- 일반 대화 쌍
  [user] + [assistant(tool_use)] + [tool_result] -- 도구 호출 쌍
```

**Step 8과의 관계**: 여기서는 단순히 오래된 메시지를 삭제합니다. Step 8에서는 삭제 대신 LLM이 요약하여 정보 손실을 최소화하는 방법을 배웁니다.

---

## Section 6: Python 구현 — MessageTruncator

In [ ]:
import json


def estimate_tokens(messages: list[dict]) -> int:
    """
    메시지의 토큰 수 추정 (간이 방식: 4자당 1토큰)
    
    정확한 토큰 카운팅은 tiktoken 등의 라이브러리가 필요하지만,
    truncation 판단에는 대략적인 추정으로 충분합니다.
    영어는 ~4자/토큰, 한국어는 ~2-3자/토큰이므로 4자를 기준으로 합니다.
    """
    total_chars = 0
    for msg in messages:
        content = msg.get("content", "")
        if isinstance(content, str):
            total_chars += len(content)
        elif isinstance(content, list):
            # Anthropic 형식: content가 블록 리스트인 경우
            for block in content:
                if isinstance(block, dict):
                    total_chars += len(str(block.get("text", "")))
                    total_chars += len(str(block.get("content", "")))
        # text 필드 (내부 포맷)
        total_chars += len(msg.get("text", ""))
        # tool_calls의 input도 토큰에 포함
        if "tool_calls" in msg:
            total_chars += len(json.dumps(msg["tool_calls"], ensure_ascii=False))
    return total_chars // 4


def _find_conversation_pairs(messages: list[dict]) -> list[tuple[int, int]]:
    """
    메시지 리스트에서 user-assistant 쌍의 인덱스를 찾습니다.
    tool_result 메시지는 직전 assistant(tool_use)와 함께 하나의 쌍에 포함됩니다.
    
    반환: [(start_idx, end_idx), ...] — 각 쌍의 시작/끝 인덱스
    """
    pairs = []
    i = 0
    while i < len(messages):
        msg = messages[i]
        if msg.get("role") == "user":
            start = i
            # 다음 assistant 메시지 찾기
            j = i + 1
            if j < len(messages) and messages[j].get("role") == "assistant":
                # assistant 뒤에 tool_result가 이어지면 포함
                end = j
                k = j + 1
                while k < len(messages) and messages[k].get("role") in ("tool_result", "tool"):
                    end = k
                    k += 1
                pairs.append((start, end))
                i = end + 1
            else:
                i += 1
        else:
            i += 1
    return pairs


def truncate_messages(messages: list[dict], max_messages: int = 20) -> list[dict]:
    """
    user-assistant 쌍 단위로 오래된 메시지를 제거합니다.
    
    Args:
        messages: 대화 히스토리
        max_messages: 유지할 최대 메시지 수
    
    Returns:
        truncation된 메시지 리스트
    """
    if len(messages) <= max_messages:
        return messages[:]

    pairs = _find_conversation_pairs(messages)
    if not pairs:
        return messages[-max_messages:]

    # 마지막부터 max_messages 이내에 들어올 때까지 쌍을 제거
    kept_messages = messages[:]
    for start, end in pairs:
        if len(kept_messages) <= max_messages:
            break
        # 쌍에 해당하는 메시지를 제거 (인덱스 조정 필요 — 앞에서부터 제거)
        # 단순화: 앞에서부터 쌍 단위로 제거
        pass

    # 간단한 구현: 쌍 단위로 앞에서부터 제거
    remove_count = 0
    for start, end in pairs:
        pair_size = end - start + 1
        if len(messages) - remove_count <= max_messages:
            break
        remove_count += pair_size

    truncated = messages[remove_count:]

    # 시작 메시지가 user가 아니면 조정 (API 규칙: 첫 메시지는 user)
    while truncated and truncated[0].get("role") not in ("user",):
        truncated = truncated[1:]

    return truncated


def truncate_by_tokens(messages: list[dict], max_tokens: int) -> list[dict]:
    """
    토큰 수 기준으로 오래된 메시지를 제거합니다.
    
    Args:
        messages: 대화 히스토리
        max_tokens: 최대 허용 토큰 수
    
    Returns:
        truncation된 메시지 리스트
    """
    current_tokens = estimate_tokens(messages)
    if current_tokens <= max_tokens:
        return messages[:]

    pairs = _find_conversation_pairs(messages)
    if not pairs:
        # 쌍을 찾을 수 없으면 뒤에서부터 유지
        result = []
        token_sum = 0
        for msg in reversed(messages):
            msg_tokens = estimate_tokens([msg])
            if token_sum + msg_tokens > max_tokens:
                break
            result.insert(0, msg)
            token_sum += msg_tokens
        return result

    # 앞에서부터 쌍을 제거하면서 토큰 수를 줄임
    remove_indices = set()
    for start, end in pairs:
        if current_tokens <= max_tokens:
            break
        for idx in range(start, end + 1):
            pair_tokens = estimate_tokens([messages[idx]])
            current_tokens -= pair_tokens
            remove_indices.add(idx)

    truncated = [msg for i, msg in enumerate(messages) if i not in remove_indices]

    # 시작 메시지가 user가 아니면 조정
    while truncated and truncated[0].get("role") not in ("user",):
        truncated = truncated[1:]

    return truncated


print("truncation 함수 정의 완료: estimate_tokens, truncate_messages, truncate_by_tokens")

In [ ]:
# ─── 데모: 긴 대화를 truncation 하기 ──────────────────────────────

def generate_tool_conversation(num_turns: int) -> list[dict]:
    """도구 호출이 포함된 긴 대화를 생성합니다."""
    messages = []
    for i in range(num_turns):
        messages.append({"role": "user", "content": f"작업 {i+1}을 수행해주세요. " + "상세한 설명 " * 20})
        messages.append({
            "role": "assistant",
            "text": f"작업 {i+1}을 처리합니다.",
            "tool_calls": [{"id": f"call_{i}", "name": "Bash", "input": {"command": f"echo task_{i}"}}],
        })
        messages.append({
            "role": "tool_result",
            "tool_use_id": f"call_{i}",
            "content": f"task_{i} 실행 결과: " + "출력 데이터 " * 50,
        })
    # 마지막 응답
    messages.append({"role": "user", "content": "모든 작업의 결과를 요약해주세요."})
    messages.append({"role": "assistant", "content": "모든 작업이 완료되었습니다."})
    return messages


# 20턴 대화 생성
long_chat = generate_tool_conversation(20)
original_tokens = estimate_tokens(long_chat)

print(f"=== Truncation 데모 ===")
print(f"원본: {len(long_chat)}개 메시지, ~{original_tokens:,} 토큰")
print()

# 1. 메시지 수 기준 truncation
truncated_by_count = truncate_messages(long_chat, max_messages=15)
print(f"[메시지 수 기준] max_messages=15")
print(f"  결과: {len(truncated_by_count)}개 메시지, ~{estimate_tokens(truncated_by_count):,} 토큰")
print(f"  첫 메시지: {truncated_by_count[0]['role']}: {truncated_by_count[0].get('content', truncated_by_count[0].get('text', ''))[:50]}...")
print()

# 2. 토큰 수 기준 truncation
truncated_by_tokens = truncate_by_tokens(long_chat, max_tokens=2000)
print(f"[토큰 수 기준] max_tokens=2000")
print(f"  결과: {len(truncated_by_tokens)}개 메시지, ~{estimate_tokens(truncated_by_tokens):,} 토큰")
if truncated_by_tokens:
    print(f"  첫 메시지: {truncated_by_tokens[0]['role']}: {truncated_by_tokens[0].get('content', truncated_by_tokens[0].get('text', ''))[:50]}...")
print()

# 쌍 보존 확인
print("=== 쌍 보존 확인 ===")
roles = [m.get('role') for m in truncated_by_count]
print(f"  메시지 역할 순서: {roles[:10]}...")
print(f"  첫 메시지가 user? {roles[0] == 'user'}")

# tool_result가 고아가 되지 않았는지 확인
orphan_results = []
for i, msg in enumerate(truncated_by_count):
    if msg.get('role') == 'tool_result':
        tool_id = msg.get('tool_use_id')
        # 직전 assistant에 해당 tool_call이 있는지 확인
        found = False
        for j in range(i - 1, -1, -1):
            if truncated_by_count[j].get('role') == 'assistant':
                tool_calls = truncated_by_count[j].get('tool_calls', [])
                if any(tc.get('id') == tool_id for tc in tool_calls):
                    found = True
                break
        if not found:
            orphan_results.append(tool_id)

print(f"  고아 tool_result 수: {len(orphan_results)} {'(정상!)' if not orphan_results else '(문제 발생!)'}")

**관찰 포인트:**
- 쌍 단위 제거로 API 규칙(user-assistant 교대)을 유지합니다
- `tool_result`는 해당 `assistant`(tool_use)와 함께 제거되므로 고아가 생기지 않습니다
- 단순 truncation은 정보 손실이 크지만, 구현이 간단하고 LLM 호출이 필요 없습니다
- Step 8에서는 삭제 대신 **LLM 요약**으로 정보를 보존하는 방법을 배웁니다

---

## Section 7: 실습 — 전체 통합

Step 1의 에이전트 루프에 CostTracker와 StreamAnalyzer를 통합합니다.
매 턴마다 비용을 추적하고, 스트리밍 통계를 수집하는 `run_agent_with_tracking()`을 만듭니다.

In [ ]:
from mini_claude.llm_client import LLMResponse
from mini_claude.agent_loop import agent_loop, AgentEvent


async def run_agent_with_tracking(
    *,
    client,
    prompt: str,
    system: str = "",
    tools: list[dict] | None = None,
    tool_executor=None,
    model_name: str = "claude-sonnet-4-20250514",
    max_turns: int = 10,
) -> dict:
    """
    에이전트 루프를 실행하면서 비용과 스트리밍 통계를 추적합니다.
    
    Returns:
        {
            'final_text': str,
            'cost_report': str,
            'turns': int,
            'turn_details': [...]
        }
    """
    tracker = CostTracker(model=model_name)
    messages = [{"role": "user", "content": prompt}]
    final_text = ""
    turn_details = []

    async for event in agent_loop(
        client=client,
        messages=messages,
        system=system,
        tools=tools,
        tool_executor=tool_executor,
        max_turns=max_turns,
    ):
        if event.type == "turn_start" and event.state:
            current_turn = event.state.turn_count
            print(f"\n{'─' * 40} Turn {current_turn} {'─' * 40}")

        elif event.type == "text":
            final_text = event.text
            print(f"  [텍스트] {event.text[:100]}{'...' if len(event.text) > 100 else ''}")

            # 응답의 토큰 사용량을 CostTracker에 기록
            if event.response:
                usage = UsageRecord(
                    input_tokens=event.response.input_tokens,
                    output_tokens=event.response.output_tokens,
                )
                cost_info = tracker.add_usage(usage)
                turn_details.append(cost_info)
                print(f"  [비용] 이번 턴: ${cost_info['turn_cost_usd']:.4f}, "
                      f"누적: ${cost_info['total_cost_usd']:.4f}")

        elif event.type == "tool_call":
            tc = event.tool_call
            print(f"  [도구] {tc.name}({tc.input})")

        elif event.type == "tool_result":
            print(f"  [결과] {event.tool_result[:80]}{'...' if len(event.tool_result) > 80 else ''}")

        elif event.type == "done":
            print(f"\n  [완료] transition: {event.state.transition if event.state else 'N/A'}")

    print("\n" + tracker.format_report())

    return {
        'final_text': final_text,
        'cost_report': tracker.format_report(),
        'cost_summary': tracker.get_summary(),
        'turns': len(turn_details),
        'turn_details': turn_details,
    }


print("run_agent_with_tracking() 정의 완료")

In [ ]:
# ─── 실제 API로 실행 (API 키가 설정된 경우) ─────────────────────

import datetime

# 도구 정의
try:
    from mini_claude.llm_client import create_client, AnthropicClient, tool_to_anthropic_schema, tool_to_openai_schema

    client = create_client()

    if isinstance(client, AnthropicClient):
        tools_schema = [
            tool_to_anthropic_schema(
                "get_current_time",
                "현재 시간을 반환합니다.",
                {"type": "object", "properties": {
                    "timezone": {"type": "string", "description": "시간대 (기본: UTC)"}
                }, "required": []}
            ),
            tool_to_anthropic_schema(
                "calculate",
                "수학 표현식을 계산합니다.",
                {"type": "object", "properties": {
                    "expression": {"type": "string", "description": "수학 표현식"}
                }, "required": ["expression"]}
            ),
        ]
    else:
        tools_schema = [
            tool_to_openai_schema(
                "get_current_time",
                "현재 시간을 반환합니다.",
                {"type": "object", "properties": {
                    "timezone": {"type": "string", "description": "시간대 (기본: UTC)"}
                }, "required": []}
            ),
            tool_to_openai_schema(
                "calculate",
                "수학 표현식을 계산합니다.",
                {"type": "object", "properties": {
                    "expression": {"type": "string", "description": "수학 표현식"}
                }, "required": ["expression"]}
            ),
        ]

    async def execute_tool(name: str, args: dict) -> str:
        if name == "get_current_time":
            tz = args.get("timezone", "UTC")
            now = datetime.datetime.now(datetime.timezone.utc)
            return f"현재 시간 ({tz}): {now.isoformat()}"
        elif name == "calculate":
            expr = args.get("expression", "")
            try:
                result = eval(expr, {"__builtins__": {}})
                return f"{expr} = {result}"
            except Exception as e:
                return f"계산 오류: {e}"
        return f"알 수 없는 도구: {name}"

    # 비용 추적과 함께 에이전트 실행
    result = await run_agent_with_tracking(
        client=client,
        prompt="지금 몇 시야? 그리고 현재 분(minute)의 제곱을 계산해줘.",
        system="도구를 사용하여 사용자를 도우세요. 한국어로 답변하세요.",
        tools=tools_schema,
        tool_executor=execute_tool,
        model_name=client.model if hasattr(client, 'model') else 'unknown',
    )

    print(f"\n=== 실행 요약 ===")
    print(f"총 턴 수: {result['turns']}")
    print(f"총 비용: ${result['cost_summary']['total_cost_usd']:.4f}")

except ValueError as e:
    print(f"API 키가 설정되지 않았습니다: {e}")
    print("ANTHROPIC_API_KEY 또는 OPENAI_API_KEY 환경변수를 설정한 후 다시 실행하세요.")
    print()
    print("--- 시뮬레이션 결과로 대체 ---")
    print()
    # API 없이도 CostTracker가 동작하는 것을 확인
    sim_tracker = CostTracker(model='claude-sonnet-4-20250514')
    sim_tracker.add_usage(UsageRecord(input_tokens=1500, output_tokens=200))
    sim_tracker.add_usage(UsageRecord(input_tokens=3200, output_tokens=350, cache_read_tokens=1200))
    sim_tracker.add_usage(UsageRecord(input_tokens=5800, output_tokens=500, cache_read_tokens=3000))
    print(sim_tracker.format_report())

---

## 핵심 설계 원칙 정리

### 1. 비용 가시성 (Cost Visibility)
모든 API 호출의 토큰 사용량과 비용을 실시간으로 추적합니다. Claude Code의 `/cost` 커맨드는 세션 전체 비용을 모델별로 분류하여 보여줍니다. 비용이 보이지 않으면 최적화할 수 없습니다.

### 2. 스트리밍 우선 (Streaming-first)
응답이 완성될 때까지 기다리지 않고, 토큰이 생성될 때마다 실시간으로 처리합니다. 이것이 Claude Code가 타이핑 효과를 만들고, 도구 실행 중에도 프로그레스를 보여줄 수 있는 이유입니다. TTFT(Time to First Token)가 사용자 체감 성능의 핵심입니다.

### 3. 점진적 메시지 관리
메시지 관리는 세 단계로 발전합니다:
- **단순 truncation** (이 Step) — 오래된 쌍을 삭제, 정보 손실 큼
- **LLM 요약** (Step 8) — 삭제 대신 요약하여 정보 보존
- **선택적 복원** (Step 8) — 필요한 정보만 복원

### 4. 캐시 최적화
시스템 프롬프트와 이전 대화의 공통 prefix는 캐시에서 읽힙니다. 캐시 읽기 가격은 입력 대비 90% 할인이므로, 긴 시스템 프롬프트를 공유하면 입력 토큰 비용을 크게 절감할 수 있습니다.

---

## 다음 Step 미리보기

비용 추적과 메시지 관리의 기초를 다졌습니다. Step 2에서는 이 에이전트 루프에 **정식 도구 시스템**을 연결합니다:

- **Tool 프로토콜** — 모든 도구가 따르는 통일된 인터페이스
- **build_tool() 팩토리** — 안전한 기본값으로 도구 생성을 단순화
- **ToolRegistry** — 도구를 이름으로 찾고, API 스키마를 자동 생성

현재 딕셔너리와 `if/elif`로 정의한 도구들을 체계적으로 관리하는 방법을 배웁니다.